In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

clean_dir = Path("../data/clean")
db_path = Path("../data/olist.db")

conn = sqlite3.connect(db_path)


In [2]:
tables = ["orders", "products", "order_items", "zip_coordinates", "customers", "sellers"]

read_options = {                                         
    "orders": {
        "parse_dates": ["order_purchase_timestamp", "order_approved_at",
                        "order_delivered_carrier_date", "order_delivered_customer_date",
                        "order_estimated_delivery_date"],
        "dtype": {"is_delayed": "boolean"},
    },
    "order_items": {"parse_dates": ["shipping_limit_date"]},
}

data = {}

for name in tables:
   df = pd.read_csv(clean_dir / f"{name}.csv", **read_options.get(name, {}))
   data[name] = df
   df.to_sql(name, conn, if_exists="replace", index = False)
   print(f"{name}: {len(df)} rows saved")
 

orders: 99441 rows saved
products: 32951 rows saved
order_items: 112650 rows saved
zip_coordinates: 19015 rows saved
customers: 99441 rows saved
sellers: 3095 rows saved


In [3]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)  

,name
0,orders
1,products
2,order_items
3,zip_coordinates
4,customers
5,sellers


In [4]:
for name in tables:
    n = pd.read_sql(f"SELECT count(*) AS n FROM {name}", conn)["n"].iloc[0]
    print(f"{name}: {n}")
  

orders: 99441
products: 32951
order_items: 112650
zip_coordinates: 19015
customers: 99441
sellers: 3095


In [5]:
is_delayed = pd.read_sql("SELECT COUNT(*) FROM orders WHERE is_delayed IS NULL", conn)
is_delayed

,COUNT(*)
0,2965


In [6]:
average = pd.read_sql("SELECT AVG(is_delayed) FROM orders", conn)
average

,AVG(is_delayed)
0,0.067737


In [7]:

indexes_config = {
    "orders": ["order_id", "customer_id"],
    "products": ["product_id"],
    "customers": ["customer_id", "customer_zip_code_prefix"],
    "sellers": ["seller_id"],
    "zip_coordinates": ["geolocation_zip_code_prefix"],
    "order_items": ["order_id", "product_id", "seller_id"],
}

for table, columns in indexes_config.items():
    for col in columns:
        conn.execute(
            f"CREATE INDEX IF NOT EXISTS idx_{table}_{col} ON {table} ({col})"
        )

conn.commit()

pd.read_sql("SELECT name, tbl_name FROM sqlite_master WHERE type = 'index'", conn)

,name,tbl_name
0,idx_orders_order_id,orders
1,idx_orders_customer_id,orders
2,idx_products_product_id,products
3,idx_customers_customer_id,customers
4,idx_customers_customer_zip_code_prefix,customers
5,idx_sellers_seller_id,sellers
6,idx_zip_coordinates_geolocation_zip_code_prefix,zip_coordinates
7,idx_order_items_order_id,order_items
8,idx_order_items_product_id,order_items
9,idx_order_items_seller_id,order_items


In [8]:
items_value = pd.read_sql("SELECT SUM(items_value) FROM orders", conn)
items_value

orders = data["orders"]

total = orders["items_value"].sum()

print(f"{items_value} | {total}")

   SUM(items_value)
0        13591643.7 | 13591643.7


In [9]:
pd.read_sql("SELECT strftime('%Y-%m', order_purchase_timestamp) FROM orders", conn).head()
pd.read_sql("SELECT count(*) FROM orders WHERE strftime('%Y-%m', order_purchase_timestamp) IS NULL", conn)

,count(*)
0,0


In [10]:
pd.read_sql("SELECT count(*) FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL", conn)


,count(*)
0,0


In [27]:
pd.read_sql("SELECT DISTINCT is_delayed FROM orders",conn)
#data["orders"]
#data["products"]
#data["order_items"]
data["customers"]
#data["zip_coordinates"]
#conn.close() 






,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


The six clean tables were loaded into a SQLite database (data/olist.db) with to_sql, using if_exists="replace" so that running the notebook again gives the same result instead of duplicating rows. SQLite has no date type: dates are stored as text, booleans as 0/1, and missing values as NULL.

I created indexes on every column used to join tables (named idx_table_column).